# Inference in Linear Dynamical Systems

Linear dynamical systems (LDS) are a core tool in time series processing. They are most often discussed in the discrete-time setting, which seems to limit their applicability. Turns out, however, that even when when we want to work in continuous time, we often approximate the system with discrete steps for inference and estimation. If you've studied discrete state hidden Markov models, then many of the core inference algorithms for an LDS may be familiar. One of the key algorithms is smoothing, which is typically solved using the forwards-backwards (or Baum-Welch) algorithm when using discrete states. When working with continuous states, however, there is another approach to smoothing that is simpler to implement using standard operations on multivariate normal distributions. The smoothing algorithm also suggests a simple sampling algorithm that can be used to draw samples from the posterior distribution over states. This post reviews these ideas.

## Linear Dynamical Systems

An LDS is a system that describes the progression of a state \\(x\\) over time. In this post, we'll model the state over a set of discrete times \\([1, \ldots, T]\\), and we'll denote the observation at a particular time using \\(x_k\\). The expected value of \\(x_{k+1}\\) is \\(A x_k\\).

## Key Operations

To implement the smoother, we need to pass around information in the form of probability distributions. In an LDS, everything is a multivariate normal distribution, which makes it easy to compute the distributions we need. There are two key operations: `predict` and `update`. We'll write down the math behind these operations and then implement them in Python. We can parameterize a multivariate normal using its mean and covariance, which we'll denote using \\(m\\) and \\(S\\) respectively.

The first operation we need to implement is `predict`. This function transforms a multivariate normal distribution over \\(x_k\\) into a distribution over \\(x_{k+1}\\) using the rules described the system. We can therefore think of it as a function from an LDS and multivariate normal to another multivariate normal (the linearity of the system means that the output is a multivariate normal). Since a multivariate normal is determined by its mean and covariance, we just need to figure out what these are. If we are given \\(x_k\\), then the mean and variance are
$$
\mathbb{E}[x_{k+1} \mid x_{k}, \mathcal{M}] = A x_{k}, \\
\mathbb{V}[x_{k+1} \mid x_{k}, \mathcal{M}] = P.
$$
If the distribution over \\(x_k\\) is multivariate normal with mean \\(m_k\\) and covariance \\(C_k\\), then 

In [2]:
from collections import namedtuple

import numpy as np

LDS = namedtuple('LDS', ['A', 'B', 'P', 'Q'])
MVN = namedtuple('MVN', ['m', 'C'])

def mvn_predict(mvn, lds):
    m = np.dot(lds.A, mvn.m)
    C = np.dot(lds.A, np.dot(lds.P, lds.A.T)) + lds.Q
    return MVN(m, C)